# AtOX-cPOS5 co-expression when grown in Glc-MeOH (60%-40%)

In [1]:
import cobra
import escher
import pandas as pd
import numpy as np
import json
import csv
from cobra.io import save_json_model
from cobra import Reaction
from cobra.flux_analysis.loopless import add_loopless, loopless_solution

In [3]:
# model_path = 'iMT1026v3.xml'
# model_path ='..\model\iMT1026v3.xml'
model_path ='..\model\iMT1026v3jup.xml'
model = cobra.io.read_sbml_model(model_path)

model

Name,iMT1026v3
Memory address,1df17130ed0
Number of metabolites,1706
Number of reactions,2237
Number of genes,1026
Number of groups,77
Objective expression,1.0*Ex_biomass - 1.0*Ex_biomass_reverse_5354f
Compartments,"Vacuole, Cytosol, Mitochondria, Peroxisome, Extracellular space, Endoplasmic Reticulum, Golgi Apparatus, Nucleus, Mitochondrial intermembrane space"


In [4]:
model.objective.expression

1.0*Ex_biomass - 1.0*Ex_biomass_reverse_5354f

In [5]:
model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
glyc_e,Ex_glyc,0.9538,3,100.00%
nh4_e,Ex_nh4,0.3009,0,0.00%
o2_e,Ex_o2,1.616,0,0.00%
pi_e,Ex_pi,0.0122,0,0.00%
so4_e,Ex_so4,0.002128,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
biomass_e,Ex_biomass,-0.04851,0,0.00%
co2_e,Ex_co2,-1.163,1,100.00%
h_e,Ex_h,-0.3153,0,0.00%
h2o_e,Ex_h2o,-2.735,0,0.00%


In [6]:
# Change grwoth on glycerol to growth on 60% glucose and 40% methanol
# Add reaction describing this growth

biomass_gly = model.reactions.get_by_id('BIOMASS_glyc')
biomass_gly.bounds = (0,0)
biomass_gly

carbs = model.metabolites.get_by_id('CARBOHYDRATES_c')
DNA = model.metabolites.get_by_id('DNA_c')
lipids = model.metabolites.get_by_id('LIPIDS_c')
prot = model.metabolites.get_by_id('PROTEIN_c')
RNA = model.metabolites.get_by_id('RNA_c')
atp = model.metabolites.get_by_id('atp_c')
cof = model.metabolites.get_by_id('cof_c')
h2o = model.metabolites.get_by_id('h2o_c')
adp = model.metabolites.get_by_id('adp_c')
biomass = model.metabolites.get_by_id('biomass_c')
h = model.metabolites.get_by_id('h_c')
pi = model.metabolites.get_by_id('pi_c')

biomass_glc_meoh = Reaction ('biomass_glc_meoh_60/40')
biomass_glc_meoh.name = 'Biomass composition (g/g) - 60/40 Glucose/Methanol'
biomass_glc_meoh.add_metabolites({carbs: -0.33,
                                   DNA: -0.001,
                                   lipids: -0.042,
                                   prot: -0.49,
                                   RNA: -0.058,
                                   atp: -63.85,
                                   cof: -1,
                                   h2o: -63.85,
                                   adp: 63.85,
                                   pi: 63.85,
                                   biomass: 1,
                                   h: 63.85})
biomass_glc_meoh

Reaction identifier,biomass_glc_meoh_60/40
Name,Biomass composition (g/g) - 60/40 Glucose/Methanol
Memory address,0x1df2c4939d0
Stoichiometry,0.33 CARBOHYDRATES_c + 0.001 DNA_c + 0.042 LIPIDS_c + 0.49 PROTEIN_c + 0.058 RNA_c + 63.85 atp_c + cof_c + 63.85 h2o_c --> 63.85 adp_c + biomass_c + 63.85 h_c + 63.85 pi_c 0.33 Carbohydrates + 0.001 DNA + 0.042 Lipids + 0.49 PROTEIN + 0.058 RNA + 63.85 ATP + Cofactors and small molecules + 63.85 H2O --> 63.85 ADP + Biomass + 63.85 H+ + 63.85 Phosphate
GPR,
Lower bound,0.0
Upper bound,1000.0


In [7]:
# Add new biomass reaction to the model

print (len(model.reactions))
model.add_reactions([biomass_glc_meoh])
print (len(model.reactions))

2237
2238


In [8]:
#Check the addition of the biomass reaction to the model
BIOMASS_GLCMEOH = model.reactions.get_by_id('biomass_glc_meoh_60/40')
BIOMASS_GLCMEOH

Reaction identifier,biomass_glc_meoh_60/40
Name,Biomass composition (g/g) - 60/40 Glucose/Methanol
Memory address,0x1df2c4939d0
Stoichiometry,0.33 CARBOHYDRATES_c + 0.001 DNA_c + 0.042 LIPIDS_c + 0.49 PROTEIN_c + 0.058 RNA_c + 63.85 atp_c + cof_c + 63.85 h2o_c --> 63.85 adp_c + biomass_c + 63.85 h_c + 63.85 pi_c 0.33 Carbohydrates + 0.001 DNA + 0.042 Lipids + 0.49 PROTEIN + 0.058 RNA + 63.85 ATP + Cofactors and small molecules + 63.85 H2O --> 63.85 ADP + Biomass + 63.85 H+ + 63.85 Phosphate
GPR,
Lower bound,0.0
Upper bound,1000.0


In [9]:
#Remove glycerol supply
glycerol_exchange = model.exchanges.get_by_id('Ex_glyc')
glycerol_exchange.bounds = (0,0)
glycerol_exchange

Reaction identifier,Ex_glyc
Name,Glycerol exchange
Memory address,0x1df2c66af90
Stoichiometry,glyc_e --> Glycerol -->
GPR,
Lower bound,0
Upper bound,0


In [10]:
# Add qmeoh constraints observed in chemostat cultivations from Sergi Monforte's doctoral thesis
methanol_exchange = model.exchanges.get_by_id('Ex_meoh')
methanol_exchange.bounds = (-1.24, -1.18)
methanol_exchange

Reaction identifier,Ex_meoh
Name,Methanol exchange
Memory address,0x1df2c723550
Stoichiometry,meoh_e <-- Methanol <--
GPR,
Lower bound,-1.24
Upper bound,-1.18


In [11]:
# Add qgluc constraints observed in chemostat cultivations from Sergi Monforte's doctoral thesis
glucose_exchange = model.reactions.get_by_id('Ex_glc_D')
glucose_exchange.bounds = (-0.72, -0.7)
glucose_exchange

Reaction identifier,Ex_glc_D
Name,D-Glucose exchange
Memory address,0x1df2c723690
Stoichiometry,glc_D_e <-- D-Glucose <--
GPR,
Lower bound,-0.72
Upper bound,-0.7


In [12]:
# Change the reactions for the synthesis of lipids, proteins and sterols from glycerol to those from glucose

model.reactions.get_by_id('LIPIDS_glyc').bounds = (0,0)
model.reactions.get_by_id('PROTEINS_glyc').bounds = (0,0)
model.reactions.get_by_id('STEROLS_glyc').bounds = (0,0)

# note: these reactions from glucose do not have the glucose specification
model.reactions.get_by_id('LIPIDS').bounds = (0,1000)
model.reactions.get_by_id('PROTEINS').bounds = (0,1000)
model.reactions.get_by_id('STEROLS').bounds = (0,1000)

In [13]:
# ATP maintenance requirement (NGAME = Non-Growth Associated Maintenance Energy) based on previous studies on 
# the growth of X33-ROL on Gluc/MeOH from the group (Eric's Master Thesis)

model.reactions.get_by_id('ATPM').bounds = (1.96, 1000)

In [14]:
rolAA = model.reactions.get_by_id('rolAA')
rolAA.bounds = (0,1000)
rolRNA = model.reactions.get_by_id('rolRNA')
rolRNA.bounds = (0,1000)
rolDNA = model.reactions.get_by_id('rolDNA')
rolDNA.bounds = (0,1000)
pROL = model.reactions.get_by_id('pROL')
pROL.bounds = (0,1000)
Rol_transport =  model.reactions.get_by_id('ROLt')
Rol_transport.bounds = (0,1000)
ROL_exchange = model.exchanges.get_by_id('Ex_rol')
ROL_exchange.bounds = (0.003,1000) #Lower bound calculated from Monforte's PhD thesis for growth in Glucose/MeOH 60/40

pFAB = model.reactions.get_by_id('pFAB')
pFAB.bounds = (0,0)
fabAA = model.reactions.get_by_id('fabAA')
fabAA.bounds = (0,0)
fabt = model.reactions.get_by_id('fabt')
fabt.bounds = (0,0)
fabRNA = model.reactions.get_by_id('fabRNA')
fabRNA.bounds = (0,0)
fabDNA = model.reactions.get_by_id('fabDNA')
FAB_exchange = model.exchanges.get_by_id('Ex_fab')
FAB_exchange.bounds = (0,0)


# Extra reactions that must be closed for simulations to run smoothly:

APAT2r = model.reactions.get_by_id('APAT2r')
APAT2r.bounds = (0,0) # reaction not present in Pichia, it is yet to be removed

MMSAD3 = model.reactions.get_by_id('MMSAD3')
MMSAD3.bounds = (0,0) # The reduction reaction of MSA into Acetil-CoA it is due to an unspecific effect. Reaction
# under evaluation of being kept or not.

In [15]:
#Creating the AtOX reaction
q6h2_m = model.metabolites.get_by_id('q6h2_m')
o2_m = model.metabolites.get_by_id('o2_m')
q6_m = model.metabolites.get_by_id('q6_m')
h2o_m = model.metabolites.get_by_id('h2o_m')

AtOX = Reaction('AtOX')
AtOX.name = 'Alternative oxidase from Histoplasma capsulatum'
AtOX.add_metabolites({q6h2_m: -2,
                       o2_m: -1,
                       q6_m: 2,
                       h2o_m: 2})
#AtOX.bounds = (1.5,1.75) #Aquests son els de la Natalia
AtOX.bounds = (0,0)
print(AtOX.reaction)

# from Saccharomyces. Found in Brenda ('for references in articles please use BRENDA:EC2.7.1.86')
# Also in Uniprot with ID Q06892 and ID YPL188W (Saccharomyces Genome Database)

o2_m + 2 q6h2_m --> 2 h2o_m + 2 q6_m


In [16]:
#Add HcAtOX reaction to the model
print (len(model.reactions))
model.add_reactions([AtOX])
print (len(model.reactions))

2238
2239


In [17]:
#Creating the cPOS5 reaction
atp_c = model.metabolites.get_by_id('atp_c')
nadh_c = model.metabolites.get_by_id('nadh_c')
adp_c = model.metabolites.get_by_id('adp_c')
nadph_c = model.metabolites.get_by_id('nadph_c')
h_c = model.metabolites.get_by_id('h_c')

cPOS5 = Reaction('cPOS5')
cPOS5.name = 'Cytosolic NADH Kinase from Saccharomyces cerevisiae'
cPOS5.add_metabolites({atp_c: -1, 
                           nadh_c: -1, 
                           adp_c: 1, 
                           nadph_c: 1,
                             h_c: 1})
cPOS5.bounds = (0,0)
print (cPOS5.reaction)
cPOS5

atp_c + nadh_c --> adp_c + h_c + nadph_c


Reaction identifier,cPOS5
Name,Cytosolic NADH Kinase from Saccharomyces cerevisiae
Memory address,0x1df2c139150
Stoichiometry,atp_c + nadh_c --> adp_c + h_c + nadph_c ATP + NADH --> ADP + H+ + NADPH
GPR,
Lower bound,0
Upper bound,0


In [18]:
#Add cPOS5 reaction to the model
print (len(model.reactions))
model.add_reactions([cPOS5])
print (len(model.reactions))

2239
2240


In [19]:
nadph_c = model.metabolites.get_by_id('nadph_c')
nadh_c = model.metabolites.get_by_id('nadh_c')
nad_c = model.metabolites.get_by_id('nad_c')
nadp_c = model.metabolites.get_by_id('nadp_c')

In [20]:
nadph_m = model.metabolites.get_by_id('nadph_m')
nadp_m = model.metabolites.get_by_id('nadp_m')
nadh_m = model.metabolites.get_by_id('nadh_m')
nad_m = model.metabolites.get_by_id('nad_m')

## Reaction Ratios as constraints

### The experimental data was extracted from the paper:
#### "Metabolic flux analysis of recombinant Pichia pastoris growing on different glycerol/methanol mixtures by iterative fitting of NMR-derived 13C labelling data from proteinogenic amino acids" by Joel Jordà, 2014

In [21]:
ArabitolSecretion = model.exchanges.get_by_id('Ex_abt_D')
ICL_Reaction = model.reactions.get_by_id('ICLx')
FBA_Reaction = model.reactions.get_by_id('FBA')
MAE2m_Reaction = model.reactions.get_by_id('ME2m')
MAE1m_Reaction = model.reactions.get_by_id('ME1m')
MAE1_Reaction = model.reactions.get_by_id('ME1')
MALSp_Reaction = model.reactions.get_by_id('MALSp')
CSm_Reaction = model.reactions.get_by_id('CSm')
ALCD19_Reaction = model.reactions.get_by_id('ALCD19')
MDHm_Reaction = model.reactions.get_by_id('MDHm')
PDH_Reaction = model.reactions.get_by_id('PDHcm')
NADHD_Reaction = model.reactions.get_by_id('NADH2_u6m')

In [22]:
# REACTION RATIOS PER GLUCOSA METANOL

ReactionRatio1 = model.problem.Constraint(model.reactions.CSm.flux_expression - model.reactions.ACONTm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio1)

ReactionRatio2 = model.problem.Constraint(model.reactions.AKGDam.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio2)

ReactionRatio8 = model.problem.Constraint(68*model.reactions.AKGDam.flux_expression - 46*model.reactions.CSm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio8)

ReactionRatio9 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression - model.reactions.GND.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio9)

ReactionRatio10 = model.problem.Constraint(0.8*model.reactions.MDHm.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio10)

ReactionRatio12 = model.problem.Constraint(35*model.reactions.PYK.flux_expression - 143*model.reactions.PC.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio12)

ReactionRatio13 = model.problem.Constraint(68*model.reactions.PYK.flux_expression - 143*model.reactions.PYRt2m.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio13)


ReactionRatio14 = model.problem.Constraint(40*model.reactions.GAPD.flux_expression - 145*model.reactions.FBA.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio14)


ReactionRatio16 = model.problem.Constraint(model.reactions.GAPD.flux_expression - model.reactions.PYK.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio16)

ReactionRatio20 = model.problem.Constraint(0.18*model.reactions.FALDtx.flux_expression - 0.82*model.reactions.DAS.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio20)

ReactionRatio21 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression + 0.4*model.reactions.Ex_glc_D.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio21)

## pFBA Simulation for Reference WT

In [23]:
#WT Rol producer strain reference flux distribution
pfba_WT = cobra.flux_analysis.pfba(model)

In [24]:
pfba_WT.fluxes['Ex_rol']

np.float64(0.003)

In [25]:
pfba_WT.fluxes['Ex_biomass']

np.float64(0.08043236291604378)

In [26]:
pfba_WT.fluxes['Ex_co2']

np.float64(2.362879410932908)

In [27]:
pfba_WT.fluxes['Ex_o2']

np.float64(-2.8939385889002107)

In [28]:
pfba_WT.fluxes['Ex_glc_D']

np.float64(-0.72)

In [29]:
pfba_WT.fluxes['Ex_meoh']

np.float64(-1.24)

In [30]:
pfba_WT.fluxes['ICDHxm']

np.float64(0.2678738364100957)

In [31]:
# NADPH turnover rate WT strain

positive_nadph_flux_sum_WT = 0.0
for reaction in nadph_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nadph_c)
    # No totes les reaccions tenen 1 de coeficient estequiomètric pel cofactor
    #independentment si s'està produint o consumint
    positive_nadph_flux_sum_WT += abs(flux*coef)
    

print (positive_nadph_flux_sum_WT/2)

0.8232315182222708


In [32]:
#NADH turnover rate WT strain

positive_nadh_flux_sum_WT = 0.0
for reaction in nadh_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nadh_c)
    positive_nadh_flux_sum_WT += abs(flux*coef)

print (positive_nadh_flux_sum_WT/2)

3.137622846928414


In [33]:
# NADP turnover rate WT strain

positive_nadp_flux_sum_WT = 0.0
for reaction in nadp_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nadp_c)
    positive_nadp_flux_sum_WT += abs(flux*coef)

print (positive_nadp_flux_sum_WT/2)

0.8232315986546336


In [34]:
# NAD turnover rate WT strain

positive_nad_flux_sum_WT = 0.0
for reaction in nad_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nad_c)
    positive_nad_flux_sum_WT += abs(flux*coef)

print (positive_nad_flux_sum_WT/2)

3.1376230077931395


## MOMA Simulations

### Carbon Source: 60%Glc/40%MeOH

In [35]:
cPOS5.bounds = (0, 0)
AtOX.bounds = (0.1, 0.1)
pfba_WT_AtOX = cobra.flux_analysis.pfba(model) 

In [36]:
# Remove Reaction Ratio Constraints before starting MOMA simulations

ReactionRatioList = [ReactionRatio1, ReactionRatio2, ReactionRatio8, ReactionRatio9, ReactionRatio10,
                     ReactionRatio12, ReactionRatio13, ReactionRatio14, ReactionRatio16, ReactionRatio20, ReactionRatio21]
                    

model.remove_cons_vars(ReactionRatioList)

In [37]:
cPOS5range = (0.02, 0.04, 0.06, 0.08, 0.1, 0.12, 0.14, 0.16, 0.18, 0.2)

In [38]:
#MOMA Simulations Glucose 60 MeOH 40 Overexpressing AtOX
MomaResults = []
for x in cPOS5range:
        AtOX.bounds = (0.1, 0.1)
        cPOS5.bounds = (x, x)
        
        # Perform MOMA
        
        moma_result = cobra.flux_analysis.moma(model,pfba_WT_AtOX,0)
        MomaResults.append(moma_result.fluxes)
        
        print(cPOS5.bounds, moma_result.fluxes['Ex_rol'], moma_result.fluxes['Ex_biomass'], moma_result.fluxes['Ex_o2'], moma_result.fluxes['Ex_co2'])

(0.02, 0.02) 0.0032292041772608695 0.07898538308086088 -2.9326503258535035 2.4004684700467065
(0.04, 0.04) 0.00346707669716862 0.07849916219959453 -2.9330795251379245 2.4007366558034433
(0.06, 0.06) 0.0037068563065793363 0.07800922251916372 -2.933531432898445 2.4010224530300417
(0.08, 0.08) 0.003946417887993169 0.07751934853834463 -2.933983271635427 2.401307872258604
(0.1, 0.1) 0.0041873503521189 0.07702764763298399 -2.9344319198996063 2.4015975189670806
(0.12, 0.12) 0.004428917883666804 0.07653410043606576 -2.934881786692641 2.4018913488532405
(0.14, 0.14) 0.004671242761792017 0.07603787243672872 -2.9353546451931645 2.4021974145629796
(0.16, 0.16) 0.004916183386539724 0.07553415578269447 -2.9358957992848587 2.40254645307976
(0.18, 0.18) 0.005161167387565438 0.07502954376819858 -2.9364536257613394 2.402905430804476
(0.2, 0.2) 0.005406359566074123 0.07452434984544783 -2.937012125691153 2.4032642693417747


In [39]:
#NonLinear MOMA Simulations Glucose 60 MeOH 40 Overexpressing AtOX and POS5 Redox cofactors exploration
MomaResults = []
for x in cPOS5range:
        cPOS5.bounds = (x, x)
        
         # Perform MOMA
        moma_result = cobra.flux_analysis.moma(model,pfba_WT_AtOX,0)
        MomaResults.append(moma_result.fluxes)
        
        # Calculate the sum of positive fluxes involving nadph_c
        positive_nadph_flux_sum_moma = 0.0
        positive_nadh_flux_sum_moma = 0.0
        positive_nadp_flux_sum_moma = 0.0
        positive_nad_flux_sum_moma = 0.0
        
        for reaction in nadph_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadph_flux_moma = moma_result.fluxes[reaction.id]
            coef1 = reaction.get_coefficient(nadph_c)
            
            # Add the flux to the sum if it's positive
            
            positive_nadph_flux_sum_moma += abs(nadph_flux_moma*coef1)
                
        for reaction in nadh_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadh_flux_moma = moma_result.fluxes[reaction.id]
            coef2 = reaction.get_coefficient(nadh_c)
            # Add the flux to the sum if it is positive
            
            positive_nadh_flux_sum_moma += abs(nadh_flux_moma*coef2)
                
        for reaction in nadp_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadp_flux_moma = moma_result.fluxes[reaction.id]
            coef3 = reaction.get_coefficient(nadp_c)
            # Add the flux to the sum if it's positive
            
            positive_nadp_flux_sum_moma += abs(nadp_flux_moma*coef3)
                
        for reaction in nad_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nad_flux_moma = moma_result.fluxes[reaction.id]
            coef4 = reaction.get_coefficient(nad_c)
            # Add the flux to the sum if it is positive
            positive_nad_flux_sum_moma += abs(nad_flux_moma*coef4)
        
        print(cPOS5.bounds, positive_nadph_flux_sum_moma/2, positive_nadh_flux_sum_moma/2, positive_nadp_flux_sum_moma/2, positive_nad_flux_sum_moma/2)

(0.02, 0.02) 0.8355862482430804 3.137006863249378 0.835586248243082 3.1370068632493795
(0.04, 0.04) 0.8518198147744799 3.1491015296950273 0.8518198147744807 3.14910160819418
(0.06, 0.06) 0.8678727707782151 3.161789182910254 0.8678727707782146 3.1617891829102533
(0.08, 0.08) 0.883925083723471 3.174476156520625 0.8839250837234702 3.1744761565206243
(0.1, 0.1) 0.8998952943041223 3.1875879644529177 0.899895294304121 3.1875881185082116
(0.12, 0.12) 0.915767309704742 3.201118675090539 0.9157673097047384 3.2011187516246387
(0.14, 0.14) 0.9319286072665813 3.2144981479613 0.9319286072665812 3.2144982239991715
(0.16, 0.16) 0.9488503362339074 3.2280164309978865 0.9488503362339071 3.2280165065320405
(0.18, 0.18) 0.9657250490978726 3.241744135272244 0.9657250490978794 3.2417442103017855
(0.2, 0.2) 0.9826007873275664 3.2554400017735814 0.9826007873275648 3.2554400762979303


In [41]:
#NonLinear MOMA Simulations Glucose 60 MeOH 40 Overexpressing AtOX and POS5 Redox cofactors exploration
#Using initial pfba_WT as reference
MomaResults = []
for x in cPOS5range:
        cPOS5.bounds = (x, x)
        AtOX.bounds = (0.1, 0.1)
        
         # Perform MOMA
        moma_result = cobra.flux_analysis.moma(model,pfba_WT,0)
        MomaResults.append(moma_result.fluxes)
        
        # Calculate the sum of positive fluxes involving nadph_c
        positive_nadph_flux_sum_moma = 0.0
        positive_nadh_flux_sum_moma = 0.0
        positive_nadp_flux_sum_moma = 0.0
        positive_nad_flux_sum_moma = 0.0
        
        for reaction in nadph_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadph_flux_moma = moma_result.fluxes[reaction.id]
            coef1 = reaction.get_coefficient(nadph_c)
            
            # Add the flux to the sum if it's positive
            
            positive_nadph_flux_sum_moma += abs(nadph_flux_moma*coef1)
                
        for reaction in nadh_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadh_flux_moma = moma_result.fluxes[reaction.id]
            coef2 = reaction.get_coefficient(nadh_c)
            # Add the flux to the sum if it is positive
            
            positive_nadh_flux_sum_moma += abs(nadh_flux_moma*coef2)
                
        for reaction in nadp_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nadp_flux_moma = moma_result.fluxes[reaction.id]
            coef3 = reaction.get_coefficient(nadp_c)
            # Add the flux to the sum if it's positive
            
            positive_nadp_flux_sum_moma += abs(nadp_flux_moma*coef3)
                
        for reaction in nad_c.reactions:
            # Get the flux value for this reaction from the MOMA result
            nad_flux_moma = moma_result.fluxes[reaction.id]
            coef4 = reaction.get_coefficient(nad_c)
            # Add the flux to the sum if it is positive
            positive_nad_flux_sum_moma += abs(nad_flux_moma*coef4)
        
        print(cPOS5.bounds, positive_nadph_flux_sum_moma/2, positive_nadh_flux_sum_moma/2, positive_nadp_flux_sum_moma/2, positive_nad_flux_sum_moma/2)

(0.02, 0.02) 0.8571197820913327 3.1680487258947423 0.857119782091332 3.168048802753183
(0.04, 0.04) 0.8714666703505164 3.1794783473025974 0.8714666703505146 3.1794784241132423
(0.06, 0.06) 0.8857849663356777 3.191213094454308 0.8857849663356864 3.191213171217898
(0.08, 0.08) 0.9000120951678304 3.203024431949011 0.9000120951678254 3.2030245086658766
(0.1, 0.1) 0.9141192889478504 3.215478027124629 0.9141192889478477 3.215478103787641
(0.12, 0.12) 0.9283442870266627 3.228269060790395 0.9283442870266618 3.2282691373940917
(0.14, 0.14) 0.9441657939169736 3.2402650857500386 0.9441657939169724 3.2402651622889693
(0.16, 0.16) 0.9601211445919094 3.2527885371144216 0.9601211445919071 3.252788613591197
(0.18, 0.18) 0.9761104481535448 3.265481601201309 0.9761104481535501 3.2654816776116204
(0.2, 0.2) 0.9920819829918848 3.2793311198311836 0.9920819829918859 3.2793311960957197
